***

## NLP - PRACTICAL 2 - VECTORIZATION

***

## aim:

To represent text numerically using Bag of Words, TF-IDF, and N-grams, and use cosine similarity to compare documents.

## problem statement:

Given a collection of text documents, preprocess the corpus and convert it into numerical vectors using Bag of Words and TF-IDF. Generate unigrams/bigrams and calculate document similarity using cosine similarity.

***

## 1. importing libraries

In [2]:
# Libraries for text processing, vectorization and similarity
import numpy as np
import pandas as pd
from collections import Counter
import math

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Shinde\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Shinde\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Shinde\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 2. data handling

In [3]:
# A corpus is simply a collection of documents
corpus = [
    "I like this movie",
    "I do not like this movie",
    "The movie was exciting",
    "The match was exciting"
]

print("Number of documents:", len(corpus))

Number of documents: 4


## 3. preprocessing

In [4]:
# Lowercase, tokenize and remove stopwords/punctuation
stop_words = set(stopwords.words("english"))

def preprocess(text):
    tokens = word_tokenize(text.lower())
    return [
        token
        for token in tokens
        if token.isalpha() and token not in stop_words
    ]

processed_corpus = [preprocess(doc) for doc in corpus]

for i, doc in enumerate(processed_corpus):
    print(f"Doc {i}:", doc)

Doc 0: ['like', 'movie']
Doc 1: ['like', 'movie']
Doc 2: ['movie', 'exciting']
Doc 3: ['match', 'exciting']


## 4. bag of words — manual

In [5]:
# Build vocabulary from all processed documents
vocab = sorted( set(word for doc in processed_corpus for word in doc))

# Count how many times each vocabulary word occurs in each document
def bow_vector(doc_tokens, vocab):
    counts = Counter(doc_tokens)
    return [counts[word] for word in vocab]

bow_matrix = [
    bow_vector(doc, vocab)
    for doc in processed_corpus
]

bow_df = pd.DataFrame(
    bow_matrix,
    columns=vocab
)

print(bow_df)

   exciting  like  match  movie
0         0     1      0      1
1         0     1      0      1
2         1     0      0      1
3         1     0      1      0


## 5. bag of words — sklearn

In [6]:
# CountVectorizer performs the same representation automatically
bow_vectorizer = CountVectorizer(stop_words="english")

bow_matrix = bow_vectorizer.fit_transform(corpus)

bow_df = pd.DataFrame( bow_matrix.toarray(),  
                      columns=bow_vectorizer.get_feature_names_out())

print(bow_df)

   exciting  like  match  movie
0         0     1      0      1
1         0     1      0      1
2         1     0      0      1
3         1     0      1      0


## 6. tf-idf — manual

In [7]:
# TF = word frequency within a document
def compute_tf(doc_tokens):
    counts = Counter(doc_tokens)
    total = len(doc_tokens)

    return {
        word: count / total
        for word, count in counts.items()
    }

# IDF gives lower weight to words appearing in many documents
def compute_idf(processed_corpus, vocab):
    N = len(processed_corpus)
    idf = {}

    for word in vocab:
        docs_containing = sum(
            1 for doc in processed_corpus
            if word in doc
        )
        idf[word] = math.log(
            N / (1 + docs_containing)
        )

    return idf

tf_list = [
    compute_tf(doc)
    for doc in processed_corpus
]

idf = compute_idf(processed_corpus, vocab)

def compute_tfidf(tf, idf, vocab):
    return [
        tf.get(word, 0) * idf[word]
        for word in vocab
    ]

tfidf_matrix = [
    compute_tfidf(tf, idf, vocab)
    for tf in tf_list
]

print(pd.DataFrame(tfidf_matrix, columns=vocab).round(3))

   exciting   like  match  movie
0     0.000  0.144  0.000    0.0
1     0.000  0.144  0.000    0.0
2     0.144  0.000  0.000    0.0
3     0.144  0.000  0.347    0.0


## 7. tf-idf — sklearn

In [8]:
# TfidfVectorizer converts documents directly into TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame( tfidf_matrix.toarray(),
                        columns=tfidf_vectorizer.get_feature_names_out() )

print(tfidf_df.round(3))

   exciting   like  match  movie
0     0.000  0.777  0.000  0.629
1     0.000  0.777  0.000  0.629
2     0.777  0.000  0.000  0.629
3     0.619  0.000  0.785  0.000


## 8. n-grams

In [9]:
# Unigram = one word; bigram = two consecutive words
uni_vectorizer = CountVectorizer( stop_words="english", ngram_range=(1, 1))

bi_vectorizer = CountVectorizer(stop_words="english", ngram_range=(2, 2))

uni_vectorizer.fit(corpus)
bi_vectorizer.fit(corpus)

print("Unigrams:")
print(uni_vectorizer.get_feature_names_out())

print("\nBigrams:")
print(bi_vectorizer.get_feature_names_out())

Unigrams:
['exciting' 'like' 'match' 'movie']

Bigrams:
['like movie' 'match exciting' 'movie exciting']


## 9. why bigrams matter

In [10]:
# Unigrams do not preserve word order
demo = [
    "I like this movie",
    "I do not like this movie"
]

uni = CountVectorizer(ngram_range=(1, 1))
bi = CountVectorizer(ngram_range=(2, 2))

print("Unigrams:")
print(pd.DataFrame(
    uni.fit_transform(demo).toarray(),
    columns=uni.get_feature_names_out()
))

print("\nBigrams:")
print(pd.DataFrame(
    bi.fit_transform(demo).toarray(),
    columns=bi.get_feature_names_out()
))

Unigrams:
   do  like  movie  not  this
0   0     1      1    0     1
1   1     1      1    1     1

Bigrams:
   do not  like this  not like  this movie
0       0          1         0           1
1       1          1         1           1


## 10. document similarity

In [11]:
# Cosine similarity compares the direction of two TF-IDF vectors
similarity_matrix = cosine_similarity(tfidf_matrix)

print(
    pd.DataFrame(
        similarity_matrix,
        index=[f"Doc {i}" for i in range(len(corpus))],
        columns=[f"Doc {i}" for i in range(len(corpus))]
    ).round(2))

# Ignore self-similarity and find the most similar pair
np.fill_diagonal(similarity_matrix, 0)

i, j = np.unravel_index(
    similarity_matrix.argmax(),
    similarity_matrix.shape)

print(f"Most similar documents: Doc {i} and Doc {j}")
print("Similarity:", round(similarity_matrix[i, j], 3))

       Doc 0  Doc 1  Doc 2  Doc 3
Doc 0    1.0    1.0   0.40   0.00
Doc 1    1.0    1.0   0.40   0.00
Doc 2    0.4    0.4   1.00   0.48
Doc 3    0.0    0.0   0.48   1.00
Most similar documents: Doc 0 and Doc 1
Similarity: 1.0


***

## output:

The program produces:
- preprocessed documents
- Bag of Words vectors
- TF-IDF vectors
- unigram and bigram features
- cosine similarity scores between documents

***

## result:

The text corpus was successfully converted into numerical representations using Bag of Words, TF-IDF and N-grams, and document similarity was calculated using cosine similarity.

## observation:

Bag of Words represents each document using word counts, while TF-IDF gives lower importance to words that occur across many documents. N-grams preserve limited word-order information, and cosine similarity identifies documents with similar vector representations.

***

## viva questions:

1. **What is Bag of Words?**  
   A representation in which each document is represented by counts of vocabulary words.

2. **What is TF-IDF?**  
   A weighting scheme that gives higher importance to words that are frequent in a document but less common across documents.

3. **What is TF?**  
   Term frequency: how frequently a word occurs in a document.

4. **What is IDF?**  
   Inverse document frequency: a measure that reduces the weight of words appearing in many documents.

5. **Why does a common word get low IDF?**  
   Because it provides less information for distinguishing documents.

6. **What is a unigram?**  
   A single word.

7. **What is a bigram?**  
   Two consecutive words.

8. **Why are bigrams useful?**  
   They preserve some word-order information, such as `not good`.

9. **What is `fit_transform()`?**  
   It learns the vocabulary/statistics from the input and transforms the input into vectors.

10. **What is `transform()`?**  
    It applies an already-fitted vectorizer to new text.

11. **What is cosine similarity?**  
    A measure of similarity based on the angle between two vectors.

12. **Why use cosine similarity for text?**  
    It compares vector direction and is less dependent on document length.

13. **Difference between BoW and TF-IDF?**  
    BoW uses raw counts; TF-IDF weights words according to their document importance.

14. **What is vocabulary?**  
    The set of unique terms used as features.

15. **What is a limitation of unigram BoW?**  
    It largely ignores word order and context.

***